In [1]:
import os
import torch
import numpy as np
from PIL import Image
from tqdm import tqdm
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation
import glob

# --- Configuration ---
MODEL_ID = "nvidia/segformer-b0-finetuned-cityscapes-1024-1024"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Adjust DATA_ROOT to be more flexible
DATA_ROOT = os.path.join("cityscapes")
IMG_DIR = os.path.join(DATA_ROOT, "leftImg8bit", "val")
GT_DIR = os.path.join(DATA_ROOT, "gtFine", "val")

# --- Memory optimization settings ---
MAX_SAMPLES = None  # Set to an integer (e.g. 50) to limit evaluation for testing
EVAL_SIZE = (1024, 512) # Half resolution (Width, Height) to save memory and GPU VRAM

ID2LABEL = {
    0: "road", 1: "sidewalk", 2: "building", 3: "wall", 4: "fence", 5: "pole", 
    6: "traffic light", 7: "traffic sign", 8: "vegetation", 9: "terrain", 10: "sky", 
    11: "person", 12: "rider", 13: "car", 14: "truck", 15: "bus", 16: "train", 
    17: "motorcycle", 18: "bicycle"
}

LABEL_ID_TO_TRAIN_ID = {
    7: 0, 8: 1, 11: 2, 12: 3, 13: 4, 17: 5, 19: 6, 20: 7, 21: 8, 22: 9, 
    23: 10, 24: 11, 25: 12, 26: 13, 27: 14, 28: 15, 31: 16, 32: 17, 33: 18
}

def compute_conf_matrix(pred, gt, num_classes=19):
    """Computes the confusion matrix to calculate IoU without keeping all predictions in memory."""
    mask = (gt >= 0) & (gt < num_classes)
    label = num_classes * gt[mask].astype('int') + pred[mask]
    count = np.bincount(label, minlength=num_classes**2)
    return count.reshape(num_classes, num_classes)

def evaluate_segformer_cityscapes_fast():
    print(f"Loading model: {MODEL_ID} on {DEVICE}")
    processor = SegformerImageProcessor.from_pretrained(MODEL_ID)
    model = SegformerForSemanticSegmentation.from_pretrained(MODEL_ID).to(DEVICE)
    model.eval()

    # Discover images
    search_pattern = os.path.join(IMG_DIR, "*", "*_leftImg8bit.png")
    image_paths = glob.glob(search_pattern)
    
    if not image_paths:
        print(f"No images found in {IMG_DIR}")
        print(f"Search pattern was: {search_pattern}")
        # Fallback to current directory for different environments
        alt_img_dir = os.path.join("cityscapes", "leftImg8bit", "val")
        image_paths = glob.glob(os.path.join(alt_img_dir, "*", "*_leftImg8bit.png"))
        if not image_paths:
            return
        print(f"Found images in alternative path: {alt_img_dir}")

    if MAX_SAMPLES:
        print(f"Limiting evaluation to {MAX_SAMPLES} samples.")
        image_paths = image_paths[:MAX_SAMPLES]

    conf_matrix = np.zeros((19, 19))
    print(f"Starting evaluation on {len(image_paths)} images...")

    for img_path in tqdm(image_paths):
        filename = os.path.basename(img_path)
        city = os.path.basename(os.path.dirname(img_path))
        file_id = filename.replace("_leftImg8bit.png", "")
        
        # Determine GT path based on current directory structure
        if "task_1" in img_path:
            gt_path = os.path.join(GT_DIR, city, f"{file_id}_gtFine_labelIds.png")
        else:
            gt_path = os.path.join("cityscapes", "gtFine", "val", city, f"{file_id}_gtFine_labelIds.png")

        if not os.path.exists(gt_path):
            continue

        # Load and resize (Resize to EVAL_SIZE to save memory)
        image = Image.open(img_path).convert("RGB").resize(EVAL_SIZE, Image.BILINEAR)
        target = Image.open(gt_path).resize(EVAL_SIZE, Image.NEAREST)
        
        target_np = np.array(target)
        new_target = np.full_like(target_np, 255) # ignore index
        for label_id, train_id in LABEL_ID_TO_TRAIN_ID.items():
            new_target[target_np == label_id] = train_id

        # Inference
        inputs = processor(images=image, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            # Interpolate to EVAL_SIZE
            upsampled_logits = torch.nn.functional.interpolate(
                logits,
                size=EVAL_SIZE[::-1], # (height, width)
                mode="bilinear",
                align_corners=False,
            )
            pred_labels = upsampled_logits.argmax(dim=1).squeeze(0).cpu().numpy()

        # Update confusion matrix
        conf_matrix += compute_conf_matrix(pred_labels, new_target, 19)

    # Compute metrics from confusion matrix
    true_pos = np.diag(conf_matrix)
    false_pos = np.sum(conf_matrix, axis=0) - true_pos
    false_neg = np.sum(conf_matrix, axis=1) - true_pos
    
    # Per-class IoU
    denominator = true_pos + false_pos + false_neg
    iou = np.divide(true_pos, denominator, out=np.zeros_like(true_pos, dtype=float), where=denominator != 0)
    
    # Overall metrics
    valid_classes = (denominator != 0)
    mean_iou = np.mean(iou[valid_classes])
    pixel_acc = np.sum(true_pos) / np.sum(conf_matrix)

    print("\n" + "="*40)
    print(f"Evaluation Results (Fast Mode)")
    print(f"Mean IoU: {mean_iou:.4f}")
    print(f"Overall Accuracy: {pixel_acc:.4f}")
    print("="*40)
    
    print("\nPer-class IoU:")
    for i, score in enumerate(iou):
        label_name = ID2LABEL.get(i, f"Class {i}")
        if denominator[i] > 0:
            print(f"  {label_name:15}: {score:.4f}")
        else:
            print(f"  {label_name:15}: N/A (Not present in validation set)")

if __name__ == "__main__":
    evaluate_segformer_cityscapes_fast()


Loading model: nvidia/segformer-b0-finetuned-cityscapes-1024-1024 on cuda


/home/21790159/.local/lib/python3.10/site-packages/transformers/image_processing_base.py:370: UserWarning: The following named arguments are not valid for `SegformerImageProcessor.__init__` and were ignored: 'feature_extractor_type', 'reduce_labels'
  image_processor = cls(**image_processor_dict)


Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

Starting evaluation on 500 images...


100%|██████████| 500/500 [00:53<00:00,  9.31it/s]


Evaluation Results (Fast Mode)
Mean IoU: 0.5805
Overall Accuracy: 0.9217

Per-class IoU:
  road           : 0.9620
  sidewalk       : 0.7325
  building       : 0.8611
  wall           : 0.4352
  fence          : 0.3749
  pole           : 0.3128
  traffic light  : 0.3488
  traffic sign   : 0.4731
  vegetation     : 0.8697
  terrain        : 0.5683
  sky            : 0.9118
  person         : 0.5411
  rider          : 0.2538
  car            : 0.8673
  truck          : 0.5430
  bus            : 0.5859
  train          : 0.5092
  motorcycle     : 0.3278
  bicycle        : 0.5511
